[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fedgutcor/lab-ia-intermedio/blob/main/recetas/01_primera_red.ipynb)

# 01 · Tu primera red neuronal con Keras

**M1 · Fundamentos de redes neuronales** · Laboratorio de IA · nivel intermedio

En esta receta vas a:
- ver capas, pesos y activaciones en una red real
- mirar por dentro un paso de aprendizaje: hacia adelante, gradientes y actualización
- entrenar y leer curvas de entrenamiento y validación
- probar una red convolucional (CNN) básica

**Cómo usarla:** en Colab elige *Entorno de ejecución → Ejecutar todo*. No hace falta instalar nada. Cuando termine, busca la celda **«Cambia esto»**, modifica un valor y vuelve a ejecutar.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

print("TensorFlow", tf.__version__, "· Keras", keras.__version__)
INICIO = time.time()

## 1 · Los datos

**Fashion-MNIST** (licencia MIT): 70.000 imágenes de 28 × 28 píxeles de prendas, en 10 clases. Viene incluido en Keras.

In [ ]:
(x_entreno, y_entreno), (x_prueba, y_prueba) = keras.datasets.fashion_mnist.load_data()
CLASES = ["camiseta", "pantalón", "suéter", "vestido", "abrigo", "sandalia", "camisa", "tenis", "bolso", "botín"]
x_entreno, x_prueba = x_entreno / 255.0, x_prueba / 255.0   # de 0-255 a 0-1

fig, ejes = plt.subplots(1, 8, figsize=(12, 2))
for i, eje in enumerate(ejes):
    eje.imshow(x_entreno[i], cmap="gray"); eje.set_title(CLASES[y_entreno[i]], fontsize=9); eje.axis("off")
plt.show()
print("Entrenamiento:", x_entreno.shape, "· Prueba:", x_prueba.shape)

## 2 · Cambia esto

Estos tres valores definen la red y su entrenamiento. Empieza como están; después cambia uno y compara las curvas.

In [ ]:
NEURONAS = 128      # neuronas de la capa oculta (prueba 16 o 512)
EPOCAS = 5          # pasadas completas por los datos (prueba 2 o 10)
TASA = 0.001        # tasa de aprendizaje del optimizador (prueba 0.01)

## 3 · La red: entrada, capas, pesos y salida

In [ ]:
def crear_red():
    return keras.Sequential([
        keras.Input((28, 28)),
        layers.Flatten(),                              # 28 × 28 → 784 números
        layers.Dense(NEURONAS, activation="relu"),     # capa oculta: pesos + activación
        layers.Dense(10, activation="softmax"),        # salida: una probabilidad por clase
    ])

red = crear_red()
red.summary()

## 4 · Un paso de aprendizaje, por dentro

Antes de entrenar mira **un solo paso** con 32 imágenes. Tiene tres partes:

1. **Hacia adelante:** la red predice y la pérdida mide cuánto se equivoca.
2. **Gradientes:** cuánto cambiaría la pérdida si se mueve cada peso.
3. **Actualización:** el optimizador mueve los pesos en la dirección que reduce la pérdida.

In [ ]:
x_lote, y_lote = x_entreno[:32], y_entreno[:32]
medir = keras.losses.SparseCategoricalCrossentropy()
optimizador = keras.optimizers.SGD(learning_rate=0.1)

with tf.GradientTape() as cinta:
    salida = red(x_lote)                          # 1 · hacia adelante
    perdida_antes = medir(y_lote, salida)
gradientes = cinta.gradient(perdida_antes, red.trainable_variables)        # 2 · gradientes
optimizador.apply_gradients(zip(gradientes, red.trainable_variables))      # 3 · actualización
perdida_despues = medir(y_lote, red(x_lote))
print(f"Pérdida en este lote: antes {float(perdida_antes):.3f} → después de un paso {float(perdida_despues):.3f}")

## 5 · Entrenar y leer las curvas

`fit` repite ese paso miles de veces. Se reserva un 10 % del entrenamiento como **validación**: datos que la red no usa para aprender y que sirven para ver si generaliza.

In [ ]:
red = crear_red()
red.compile(optimizer=keras.optimizers.Adam(TASA), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
historia = red.fit(x_entreno, y_entreno, epochs=EPOCAS, batch_size=128, validation_split=0.1, verbose=2)

fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.5))
a.plot(historia.history["loss"], "o-", label="entrenamiento"); a.plot(historia.history["val_loss"], "o-", label="validación")
a.set_title("Pérdida"); a.set_xlabel("época"); a.legend()
b.plot(historia.history["accuracy"], "o-", label="entrenamiento"); b.plot(historia.history["val_accuracy"], "o-", label="validación")
b.set_title("Exactitud"); b.set_xlabel("época"); b.legend()
plt.show()

**Cómo leerlas:** si la curva de entrenamiento sigue bajando y la de validación se estanca o sube, la red está memorizando (sobreajuste). Más épocas no lo arreglan: revisa la capacidad de la red o los datos.

In [ ]:
perdida, exactitud = red.evaluate(x_prueba, y_prueba, verbose=0)
print(f"Exactitud en prueba (10.000 imágenes que la red nunca vio): {exactitud:.3f}")

pred = red.predict(x_prueba[:1000], verbose=0).argmax(axis=1)
errores = np.where(pred != y_prueba[:1000])[0][:8]
fig, ejes = plt.subplots(1, len(errores), figsize=(12, 2.2))
for eje, i in zip(ejes, errores):
    eje.imshow(x_prueba[i], cmap="gray"); eje.axis("off")
    eje.set_title(f"es {CLASES[y_prueba[i]]}\ndijo {CLASES[pred[i]]}", fontsize=8)
plt.show()

## 6 · Bonus: una red convolucional (CNN)

Una CNN no aplana la imagen al principio: recorre la imagen con filtros pequeños que detectan bordes y texturas. Se entrena con los mismos datos que la red densa, en 3 épocas.

In [ ]:
cnn = keras.Sequential([
    keras.Input((28, 28, 1)),
    layers.Conv2D(16, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dense(64, activation="relu"), layers.Dense(10, activation="softmax"),
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(x_entreno[..., None], y_entreno, epochs=3, batch_size=128, verbose=2)
_, exactitud_cnn = cnn.evaluate(x_prueba[..., None], y_prueba, verbose=0)
print(f"Red densa: {exactitud:.3f} · CNN: {exactitud_cnn:.3f}")
print(f"Tiempo total de la receta: {time.time() - INICIO:.0f} s")

## Para tu proyecto

- ¿Tu tarea es de **imágenes** o de **texto**? La siguiente receta convierte texto en números.
- Anota qué pasó con las curvas cuando cambiaste un valor: esa observación es evidencia, aunque el resultado no mejore.